State Management(First Application)
1. how updates work(default: overwrite)
2. custom reducers(preview)

In [2]:
import re
from typing import TypedDict
from langgraph.graph import START, StateGraph, END

class TextState(TypedDict):
    raw_text: str
    cleaned: str
    word_count: str
    summary: str

def clean_text(state: TextState) -> dict:
    cleaned = re.sub(r'\s+', ' ', state['raw_text'].strip()).lower()
    return {'cleaned': cleaned}

def count_words(state: TextState) -> dict:
    return { 'word_count': len(state['cleaned'].split())}

def format_result(state: TextState) -> dict:
    preview = state['cleaned'][:50] + ('...' if len(state['cleaned'])> 50 else '')
    summary = ('=== Text Analysis Summary === \n'
               f"Word count   : {state['word_count']}\n"
               f"Preview      : {preview}\n"
               '========================================'
               )
    return {"summary": summary}

builder = StateGraph(TextState)

builder.add_node('clean_text',clean_text)
builder.add_node('count_words',count_words)
builder.add_node('format_result',format_result)

builder.add_edge(START, "clean_text")
builder.add_edge("clean_text", "count_words")
builder.add_edge("count_words", "format_result")
builder.add_edge("format_result", END)

graph = builder.compile()

result = graph.invoke({"raw_text": "Hey, Hi! This is Vaishnavi from Chennai. I am a professional full stack developer."})

print(result["summary"])


=== Text Analysis Summary === 
Word count   : 14
Preview      : hey, hi! this is vaishnavi from chennai. i am a pr...


State Management Deep Dive

In [3]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

class BadListState(TypedDict):
    steps: list # no reducer - defauly overwrite behaviour.

def step_one(state: BadListState) -> dict:
    current = state["steps"]
    new_list = current + ["step_one"]
    print(f" step_one returning: {new_list}")
    return {"steps": new_list}

def step_two(state: BadListState) -> dict:
    current = state["steps"]
    new_list = current + ["step_two"]
    print(f"step_two returning: {new_list}")
    return {"steps": new_list}

builder = StateGraph(BadListState)

builder.add_node("step_one", step_one)
builder.add_node("step_two", step_two)

builder.add_edge(START, "step_one")
builder.add_edge("step_one", "step_two")
builder.add_edge("step_two", END)

graph = builder.compile()

result = graph.invoke({"steps": []})

print("Final steps is:: ",result["steps"])


 step_one returning: ['step_one']
step_two returning: ['step_one', 'step_two']
Final steps is::  ['step_one', 'step_two']


This increses memory issues because it is continiously storing the values. 

There comes a solution called Reducers:

In [5]:
from typing import TypedDict, Annotated
from operator import add
from langgraph.graph import StateGraph, START, END

class GoodListState(TypedDict):
    steps: Annotated[list, add] # reducer is added here!

def step_one(state: GoodListState) -> dict:
    # current = state["steps"]
    # new_list = current + ["step_one"] - reducer will take care of the addition of the list.
    new_list = ["step_one"]
    print(f" step_one returning: {new_list}")
    return {"steps": new_list}

def step_two(state: GoodListState) -> dict:
    # current = state["steps"]
    # new_list = current + ["step_two"] = this addition is not necessary because the reducer will add it.
    new_list = ["step_two"]
    print(f"step_two returning: {new_list}")
    return {"steps": new_list}

builder = StateGraph(GoodListState)

builder.add_node("step_one", step_one)
builder.add_node("step_two", step_two)

builder.add_edge(START, "step_one")
builder.add_edge("step_one", "step_two")
builder.add_edge("step_two", END)

graph = builder.compile()

result = graph.invoke({"steps": []})

print("Final steps is:: ",result["steps"])

 step_one returning: ['step_one']
step_two returning: ['step_two']
Final steps is::  ['step_one', 'step_two']


this type of reducer will cause problems is fetching the values since it i not having any specific id's for it. 

so we need to use the reducer called "add_messages" instead of "add" reducer. This helps to add the values with ID.


We can implement that by using the message_state

In [9]:
from typing import TypedDict, Annotated
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langchain_core.messages import HumanMessage, AIMessage

class ChatState(TypedDict):
    messages: Annotated[list, add_messages]
    turn: int

def node_human_input(state: ChatState) -> dict:
    new_msg = HumanMessage(content="What is the capital of france?")
    print(f" [human_input] appending: {new_msg.content}")
    return {
        "messages": [new_msg],
        "turn": state["turn"] + 1
    }

def node_ai_reply(state: ChatState) -> dict:
    new_msg = HumanMessage(content="The capital of France is Paris")
    print(f" [ai_reply] appending: {new_msg.content}")
    return {
            "messages": [new_msg]
        }

builder = StateGraph(ChatState)

builder.add_node("human_input", node_human_input)
builder.add_node("ai_reply", node_ai_reply)

builder.add_edge(START, "human_input")
builder.add_edge("human_input", "ai_reply")
builder.add_edge("ai_reply", END)

graph = builder.compile()

result = graph.invoke({"messages": [], "turn": 0})

print(f"final result::: ", result)

 [human_input] appending: What is the capital of france?
 [ai_reply] appending: The capital of France is Paris
final result:::  {'messages': [HumanMessage(content='What is the capital of france?', additional_kwargs={}, response_metadata={}, id='c9729598-c0ac-4388-af4d-845eba54f36e'), HumanMessage(content='The capital of France is Paris', additional_kwargs={}, response_metadata={}, id='b55ffde0-b4b4-4ae5-b627-317bf5036510')], 'turn': 1}


Custom reducer:

In [10]:
def my_reducer(old_value, new_value):
    # add the combined logic here, however we want.
    return "combined_value"

In [11]:
class ChatState(TypedDict):
    messages: Annotated[list, my_reducer]
    turn: int